# 🔐 Privacy-Preserving Search: Interactive Tutorial

This notebook demonstrates:
1. How federated learning works
2. How differential privacy protects data
3. Privacy-accuracy trade-offs

**No prior setup needed** - run cells in order!

In [ ]:
# Imports
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from client.models.search_model import SearchRankingModel, extract_features
from client.differential_privacy import DifferentialPrivacy
from client.local_train import LocalClient
from server.federated_server import FederatedServer

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Imports complete!")

## Part 1: Understanding the Problem

**Traditional ML**: Collect all user data centrally
```
Users → Server (all data) → Train Model
```

**Privacy Risk**: Server sees everything!

**Federated Learning**: Data stays on device
```
Users → Local Training → Send Gradients → Server Aggregates
```

## Part 2: The AI Model

Let's see our simple neural network for click prediction.

In [ ]:
# Create model
model = SearchRankingModel(input_size=3, hidden_size=8)

print("Model Architecture:")
print(f"  Input Layer: {model.input_size} features")
print(f"  Hidden Layer: {model.hidden_size} neurons (ReLU)")
print(f"  Output Layer: 1 neuron (Sigmoid)")
print(f"\nTotal Parameters:")
print(f"  W1: {model.W1.shape} = {model.W1.size} params")
print(f"  b1: {model.b1.shape} = {model.b1.size} params")
print(f"  W2: {model.W2.shape} = {model.W2.size} params")
print(f"  b2: {model.b2.shape} = {model.b2.size} params")
print(f"  Total: {model.W1.size + model.b1.size + model.W2.size + model.b2.size}")

In [ ]:
# Test feature extraction
query = "best laptop"
doc1 = "Top 10 Best Gaming Laptops Under $1000"
doc2 = "Healthy Breakfast Recipes"

features1 = extract_features(query, doc1)
features2 = extract_features(query, doc2)

print(f"Query: '{query}'\n")
print(f"Document 1 (Relevant): '{doc1}'")
print(f"  Features: {features1}")
print(f"  [query_len, doc_len, overlap]\n")

print(f"Document 2 (Irrelevant): '{doc2}'")
print(f"  Features: {features2}")
print(f"\nNotice: Doc1 has higher overlap (0.5 vs 0.0)!")

## Part 3: Differential Privacy in Action

Let's see what happens when we add noise to gradients.

In [ ]:
# Create dummy gradients
original_gradients = {
    'W1': np.random.randn(3, 8) * 0.1,
    'b1': np.random.randn(1, 8) * 0.1,
    'W2': np.random.randn(8, 1) * 0.1,
    'b2': np.random.randn(1, 1) * 0.1
}

# Test different privacy levels
privacy_levels = ['none', 'low', 'medium', 'high']
results = []

for level in privacy_levels:
    dp = DifferentialPrivacy.from_privacy_level(level)
    noisy = dp.privatize_gradients(original_gradients)
    
    # Calculate distortion
    orig_norm = sum(np.linalg.norm(g) for g in original_gradients.values())
    noisy_norm = sum(np.linalg.norm(g) for g in noisy.values())
    distortion = abs(noisy_norm - orig_norm) / orig_norm
    
    results.append({
        'level': level,
        'sigma': dp.sigma,
        'distortion': distortion
    })

# Display results
print("Privacy Level Comparison:\n")
print(f"{'Level':<12} {'Sigma (σ)':<12} {'Distortion':<12}")
print("-" * 40)
for r in results:
    print(f"{r['level']:<12} {r['sigma']:<12} {r['distortion']:<12.1%}")

print("\nKey Insight: Higher σ = More distortion = Better privacy!")

In [ ]:
# Visualize gradient distortion
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Noise levels
sigmas = [r['sigma'] for r in results]
levels = [r['level'] for r in results]
colors = ['#27ae60', '#f39c12', '#e67e22', '#c0392b']

axes[0].bar(levels, sigmas, color=colors, alpha=0.7, edgecolor='black')
axes[0].set_ylabel('Noise Scale (σ)', fontweight='bold')
axes[0].set_title('Privacy Noise by Level', fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# Plot 2: Distortion
distortions = [r['distortion'] * 100 for r in results]
axes[1].bar(levels, distortions, color=colors, alpha=0.7, edgecolor='black')
axes[1].set_ylabel('Gradient Distortion (%)', fontweight='bold')
axes[1].set_title('Privacy Cost (Gradient Distortion)', fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Part 4: Privacy Budget (Epsilon)

Epsilon (ε) measures privacy loss:
- **Lower ε = Stronger Privacy**
- **Higher ε = Weaker Privacy**

Common thresholds:
- ε < 1: Very strong
- ε < 5: Strong
- ε < 10: Moderate
- ε > 10: Weak

In [ ]:
# Calculate privacy budgets
num_rounds = 10

print(f"Privacy Budget after {num_rounds} rounds:\n")
print(f"{'Level':<12} {'Sigma':<10} {'Epsilon (ε)':<15} {'Privacy Rating'}")
print("-" * 60)

for level in privacy_levels:
    dp = DifferentialPrivacy.from_privacy_level(level)
    summary = dp.get_privacy_summary(num_rounds)
    
    epsilon_str = str(summary['epsilon'])
    print(f"{level:<12} {summary['sigma']:<10} {epsilon_str:<15} {summary['privacy_level']}")

print("\n✓ Lower epsilon means better privacy protection!")

## Part 5: Federated Learning Simulation

Let's simulate federated learning with multiple clients!

In [ ]:
# First, generate data if not already done
import subprocess

data_dir = Path('../client/local_data')
if not (data_dir / 'device_0_data.json').exists():
    print("Generating synthetic data...")
    subprocess.run(['python', '../generate_data.py'], check=True)
    print("✓ Data generated!")
else:
    print("✓ Data already exists!")

In [ ]:
# Create federated simulation
from simulation.run_federated import FederatedSimulation

# Test with medium privacy
print("Creating federated simulation with 5 clients...\n")

sim = FederatedSimulation(
    num_clients=5,
    privacy_level='medium',
    data_dir='../client/local_data'
)

print("\n✓ Simulation ready!")

In [ ]:
# Run a few federated rounds
print("Training for 5 rounds...\n")

history = sim.train(
    num_rounds=5,
    num_local_epochs=2,
    participation_rate=1.0
)

print("\n✓ Training complete!")

In [ ]:
# Visualize training progress
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

rounds = history['rounds']
accuracy = [a * 100 for a in history['avg_accuracy']]
loss = history['avg_loss']

# Accuracy plot
ax1.plot(rounds, accuracy, marker='o', linewidth=2, markersize=8, color='#2ecc71')
ax1.set_xlabel('Federated Round', fontweight='bold')
ax1.set_ylabel('Accuracy (%)', fontweight='bold')
ax1.set_title('Model Learning Over Rounds', fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.set_ylim([0, 100])

# Loss plot
ax2.plot(rounds, loss, marker='s', linewidth=2, markersize=8, color='#e74c3c')
ax2.set_xlabel('Federated Round', fontweight='bold')
ax2.set_ylabel('Loss', fontweight='bold')
ax2.set_title('Training Loss Over Rounds', fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal Accuracy: {accuracy[-1]:.1f}%")
print(f"Final Loss: {loss[-1]:.4f}")

## Part 6: Privacy-Accuracy Trade-off

Now let's compare different privacy levels!

In [ ]:
# Compare privacy levels (this takes a few minutes)
privacy_results = {}

for level in ['none', 'low', 'medium', 'high']:
    print(f"\n{'='*60}")
    print(f"Testing: {level.upper()} Privacy")
    print(f"{'='*60}")
    
    sim = FederatedSimulation(
        num_clients=5,
        privacy_level=level,
        data_dir='../client/local_data'
    )
    
    history = sim.train(num_rounds=5, num_local_epochs=2)
    
    summary = sim.server.get_training_summary()
    privacy_summary = sim.clients[0].dp.get_privacy_summary(5)
    
    privacy_results[level] = {
        'accuracy': summary['final_accuracy'],
        'sigma': privacy_summary['sigma'],
        'epsilon': privacy_summary['epsilon']
    }

print("\n✓ All experiments complete!")

In [ ]:
# Visualize privacy-accuracy trade-off
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

levels = list(privacy_results.keys())
accuracies = [privacy_results[l]['accuracy'] * 100 for l in levels]
sigmas = [privacy_results[l]['sigma'] for l in levels]

colors = ['#27ae60', '#f39c12', '#e67e22', '#c0392b']

# Plot 1: Accuracy comparison
bars = axes[0].bar(levels, accuracies, color=colors, alpha=0.8, edgecolor='black', linewidth=2)
axes[0].set_ylabel('Accuracy (%)', fontweight='bold', fontsize=12)
axes[0].set_title('Privacy vs Accuracy', fontweight='bold', fontsize=14)
axes[0].set_ylim([0, 100])
axes[0].grid(True, alpha=0.3, axis='y')

# Add values on bars
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., height + 1,
                f'{acc:.1f}%', ha='center', fontweight='bold')

# Plot 2: Trade-off scatter
scatter = axes[1].scatter(sigmas, accuracies, s=300, c=colors, alpha=0.8,
                         edgecolors='black', linewidths=2)

for i, (sigma, acc, level) in enumerate(zip(sigmas, accuracies, levels)):
    axes[1].annotate(level.upper(), (sigma, acc),
                    xytext=(5, 5), textcoords='offset points',
                    fontweight='bold')

axes[1].set_xlabel('Privacy Noise (σ)', fontweight='bold', fontsize=12)
axes[1].set_ylabel('Accuracy (%)', fontweight='bold', fontsize=12)
axes[1].set_title('Privacy-Accuracy Trade-off', fontweight='bold', fontsize=14)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([0, 100])

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("KEY FINDING:")
print("="*60)
accuracy_drop = accuracies[0] - accuracies[-1]
print(f"Privacy has a cost: Going from 'none' to 'high' reduces")
print(f"accuracy by {accuracy_drop:.1f}% but provides strong privacy!")
print("\nRecommended: 'medium' privacy balances protection and utility.")
print("="*60)

## Summary

### What We Learned:

1. **Federated Learning** allows models to learn from distributed data without collecting it
2. **Differential Privacy** adds mathematical privacy guarantees through noise
3. **Privacy-Accuracy Trade-off** is fundamental - more privacy costs accuracy
4. **Practical Balance** exists at medium privacy levels

### Why This Matters:

This approach enables:
- ✅ AI that respects user privacy
- ✅ Compliance with regulations (GDPR, CCPA)
- ✅ User trust and transparency
- ✅ Decentralized machine learning

### Next Steps:

1. Run `python simulation/run_federated.py --mode comparison` for full analysis
2. Experiment with different privacy levels
3. Modify the model architecture
4. Read the academic papers in README

---

**This is the future of privacy-preserving AI!** 🚀